# 01 · 第一个因子探索 (momentum_20)

**目标**: 验证 momentum_20 因子在中证 1000 上的 IC / RankIC / decile 表现.

**流程**:
1. 加载 stock pool (CSI 1000)
2. 加载 adj K 线 (2020-2024)
3. 算 momentum_20 (T, N)
4. 算 forward return
5. 跑 alpha 回测 (IC / RankIC / decile)
6. 画图 + 输出结论

**前置**: `python scripts/run_fetch_noproxy.py --start 2020-01-01 --end 2024-12-31`

In [ ]:
import sys
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from alpha_backend.datas.universe import StockPool
from alpha_backend.datas.storage import load_bars_from_parquet
from alpha_backend.factors.classic import momentum  # 触发注册
from alpha_backend.factors import get as get_factor
from alpha_backend.engines.backtest_alpha import (
    forward_returns_from_prices,
    run_alpha_backtest,
)

In [ ]:
START = date(2020, 1, 1)
END = date(2024, 12, 31)
FACTOR_NAME = "momentum_20"

pool = StockPool.load()
stock_ids = pool.export()["stock_id"].tolist()
print(f"pool.size() = {pool.size()}")

bars = load_bars_from_parquet(stock_ids, start=START, end=END, kind="adj", root=PROJECT_ROOT / "datas")
print(f"bars.T = {bars.T}, bars.N = {bars.N}")

In [ ]:
factor = get_factor(FACTOR_NAME)
T, N = bars.adj_close.shape
scores = np.full((T, N), np.nan)
min_hist = (factor.spec.window or 20) + 1
for t in range(min_hist - 1, T):
    scores[t] = factor.compute(asof=bars.dates[t], prices=bars.adj_close[: t + 1])
print(f"{FACTOR_NAME} done: n_valid={int(np.sum(~np.isnan(scores)))}")

In [ ]:
fwd_ret = forward_returns_from_prices(bars.adj_close, horizon=1)
result = run_alpha_backtest(scores, fwd_ret, bars.dates, n_groups=10, min_valid=30)
print(result.summary())

In [ ]:
dates_arr = pd.DatetimeIndex(bars.dates)
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(dates_arr, result.ic, 'b-', alpha=0.7)
axes[0].axhline(result.ic_mean, color='r', ls='--', label=f'mean={result.ic_mean:.4f}')
axes[0].set_title(f"IC time series ({FACTOR_NAME})")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(dates_arr, result.rank_ic, 'g-', alpha=0.7)
axes[1].axhline(result.rank_ic_mean, color='r', ls='--', label=f'mean={result.rank_ic_mean:.4f}')
axes[1].set_title(f"RankIC time series ({FACTOR_NAME})")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

cum = np.nancumsum(result.decile_returns, axis=0)
for g in range(10):
    axes[2].plot(dates_arr, cum[:, g], label=f'G{g+1}', linewidth=1.5)
axes[2].legend(loc='upper left', ncol=5, fontsize=8)
axes[2].set_title("Decile cumulative returns")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 评价结论

根据 `result.summary()`:
- IC mean: 越接近 0 越无信号, |IC| > 0.02 表示因子有效
- ICIR (IC mean / IC std): > 0.5 表明稳定
- decile 单调: 因子 score 排序与收益正相关 → 策略可构建

M1 阶段 `momentum_20` 在 A 股短期反转效应较强, 长周期动量较弱 (这是已知的 market structure).